# 05 Evaluation
Assess performance metrics, subgroup gaps, and ethical considerations.

In [7]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

sys.path.append(str(Path('..').resolve()))
from src.evaluation import classification_metrics, metrics_by_group
from src.model import build_candidate_models, train_pipeline, predict_with_scores
from src.preprocessing import add_hit_label, make_preprocessor

df = pd.read_csv('../data/raw/spotify.csv')
df.columns = [c.lower() for c in df.columns]

if 'hit' not in df.columns:
    popularity_col = 'popularity' if 'popularity' in df.columns else 'track_popularity'
    df = add_hit_label(df, popularity_col=popularity_col, threshold=70)

leakage_cols = [
    'hit',
    'peak_position',
    'weeks_on_chart',
    'debut_rank',
    'song_display',
    'artist_display',
    'song',
    'artist',
    'track_name',
    'track_artist'
 ]

if 'first_week' in df.columns:
    df['first_week'] = pd.to_datetime(df['first_week'], errors='coerce')
    df = df.sort_values('first_week').reset_index(drop=True)
else:
    df['__time_proxy__'] = np.arange(len(df))

drop_popularity = True
if drop_popularity and 'popularity' in df.columns:
    leakage_cols.append('popularity')

preferred_features = [
    'danceability', 'energy', 'loudness', 'valence', 'tempo',
    'acousticness', 'speechiness', 'instrumentalness', 'liveness',
    'duration', 'sentiment_polarity', 'sentiment_subjectivity', 'mentions',
    'spotify_genre', 'playlist_genre', 'playlist_subgenre', 'genre'
 ]

available_features = [c for c in preferred_features if c in df.columns]
if len(available_features) == 0:
    available_features = [
        c for c in df.columns
        if c not in leakage_cols + ['first_week', '__time_proxy__']
    ]

model_df = df[available_features + ['hit']].copy()

split_idx = int(0.8 * len(df))
x_train = model_df.iloc[:split_idx].drop(columns=['hit'])
y_train = model_df.iloc[:split_idx]['hit']
x_test = model_df.iloc[split_idx:].drop(columns=['hit'])
y_test = model_df.iloc[split_idx:]['hit']

num_cols = x_train.select_dtypes(include=['number']).columns.tolist()
cat_cols = [c for c in x_train.columns if c not in num_cols]
preprocessor = make_preprocessor(num_cols, cat_cols)

C:\Users\lenovo\AppData\Local\Temp\ipykernel_12156\3311179969.py:34: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['first_week'] = pd.to_datetime(df['first_week'], errors='coerce')


In [8]:
candidate_models = build_candidate_models()

results = []
trained = {}

for model_name, estimator in candidate_models.items():
    pipeline = train_pipeline(preprocessor, estimator, x_train, y_train)
    y_pred_tmp, y_score_tmp = predict_with_scores(pipeline, x_test)
    m = classification_metrics(y_test, y_pred_tmp, y_score_tmp)
    m['model'] = model_name
    results.append(m)
    trained[model_name] = pipeline

comparison_df = pd.DataFrame(results).sort_values('f1_macro', ascending=False)
best_name = comparison_df.iloc[0]['model']
best_model = trained[best_name]
y_pred, y_score = predict_with_scores(best_model, x_test)

overall_metrics = classification_metrics(y_test, y_pred, y_score)
overall_metrics['model'] = best_name
comparison_df, overall_metrics

(   accuracy  precision_macro  recall_macro  f1_macro   roc_auc  \
 0  0.716005         0.568308      0.592293  0.572273  0.621562   
 2  0.839098         0.753445      0.537640  0.530015  0.685851   
 1  0.833503         0.684088      0.532666  0.523221  0.664885   
 
                  model  
 0  logistic_regression  
 2              xgboost  
 1        random_forest  ,
 {'accuracy': 0.7160054255679892,
  'precision_macro': 0.5683081388715706,
  'recall_macro': 0.5922927317181595,
  'f1_macro': 0.5722727127685805,
  'roc_auc': 0.621561993200135,
  'model': 'logistic_regression'})

In [9]:
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=['actual_0', 'actual_1'], columns=['pred_0', 'pred_1'])
cm_df

,pred_0,pred_1
actual_0,3821,1087
actual_1,588,402


In [10]:
eval_df = x_test.copy()
eval_df['y_true'] = y_test.values
eval_df['y_pred'] = y_pred

if 'genre' not in eval_df.columns:
    if 'spotify_genre' in eval_df.columns:
        eval_df['genre'] = eval_df['spotify_genre']
    elif 'playlist_genre' in eval_df.columns:
        eval_df['genre'] = eval_df['playlist_genre']
    else:
        eval_df['genre'] = 'unknown'

genre_metrics = metrics_by_group(eval_df, y_true_col='y_true', y_pred_col='y_pred', group_col='genre')
genre_metrics.head(15)

,group,n_samples,accuracy,f1_macro
0,21st century classical,1,1.0,1.0
2,acid jazz,2,1.0,1.0
3,acoustic pop,1,1.0,1.0
8,alternative country,1,1.0,1.0
6,afropop,2,1.0,1.0
19,appalachian folk,2,1.0,1.0
17,anthem worship,1,1.0,1.0
12,alternative metal,5,1.0,1.0
18,antiviral pop,1,1.0,1.0
42,big room,2,1.0,1.0


In [11]:
results_dir = Path('../reports/results')
results_dir.mkdir(parents=True, exist_ok=True)

pd.DataFrame([overall_metrics]).to_csv(results_dir / 'final_metrics.csv', index=False)
comparison_df.to_csv(results_dir / 'model_comparison.csv', index=False)
cm_df.to_csv(results_dir / 'confusion_matrix.csv', index=True)
genre_metrics.to_csv(results_dir / 'genre_group_metrics.csv', index=False)
eval_df[['y_true', 'y_pred', 'genre']].to_csv(results_dir / 'eval_predictions.csv', index=False)

print('Saved: final_metrics.csv, model_comparison.csv, confusion_matrix.csv, genre_group_metrics.csv, eval_predictions.csv')

Saved: final_metrics.csv, model_comparison.csv, confusion_matrix.csv, genre_group_metrics.csv, eval_predictions.csv
